# Markov por Estudante com Eixos Fixos (StudyChat)
Este notebook gera **heatmaps comparáveis entre alunos**, usando um **conjunto global fixo** de estados e superestados:

- `states` (1ª ordem): **todas** as labels observadas no dataset, ordenadas globalmente (mesmo se o aluno não passou por algumas).

- `superstates` (2ª ordem): **todos os pares (a,b)** observados no dataset (mesmo se um aluno não tiver percorrido algum par).

Assim, todos os usuários têm **mesmo shape e mesma ordem** nos eixos, permitindo comparação direta.


Saídas em `/mnt/data/markov_user_fixed_plots` e um JSON com a ordem global de eixos.



In [16]:

# ============================================================
# 1) Imports e setup
# ============================================================
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from itertools import tee

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.5f}')

out_dir = "da_artifacts"
os.makedirs(out_dir, exist_ok=True)

def pairwise(iterable):
    a, b = tee(iterable)
    next(b, None)
    return zip(a, b)

def tripletwise(iterable):
    a, b, c = tee(iterable, 3)
    next(b, None)
    next(c, None); next(c, None)
    return zip(a, b, c)

print("Diretório de saída:", out_dir)


Diretório de saída: da_artifacts


In [17]:
import numpy as np
import pandas as pd

JSON_PATH = "../studychat_supervised_da.json"  # array de objetos com da_name

# Carrega JSON
df = pd.read_json(JSON_PATH)

# Usar o nome do diálogo (da_name) como rótulo
label_col = "da_name"

# Normaliza timestamp em epoch ms (se existir)
def safe_parse_timestamp(x):
    try:
        return int(x)  # epoch ms
    except Exception:
        pass
    try:
        return pd.to_datetime(x).value // 10**6
    except Exception:
        return np.nan

if "timestamp" in df.columns:
    df["timestamp_ms"] = df["timestamp"].apply(safe_parse_timestamp)
else:
    df["timestamp_ms"] = np.nan

# interactionCount para ordenação (se existir)
df["interactionCount"] = pd.to_numeric(df.get("interactionCount", np.nan), errors="coerce")

# Remove linhas sem rótulo
df = df.dropna(subset=[label_col]).copy()

# Ordena por chat usando (timestamp_ms, interactionCount)
df["_order_key"] = list(zip(
    df["timestamp_ms"].fillna(np.inf),
    df["interactionCount"].fillna(np.inf)
))
df = df.sort_values(by=["chatId", "_order_key"]).drop(columns=["_order_key"])

# Sequências por chat (usando da_name)
seq_by_chat = (
    df.groupby(["userId", "chatId", "topic"])[label_col]
      .apply(list)
      .reset_index(name="sequence")
)

print("Conversas válidas:", len(seq_by_chat))
seq_by_chat.head(3)

Conversas válidas: 2214


,userId,chatId,topic,sequence
0,011bb520-7041-704f-b3e7-ab5c43dd3950,00078d7c-022c-429b-9e07-33459b7a5f6e,a2,"[Statement-non-opinion, Quotation, Yes-No-Ques..."
1,011bb520-7041-704f-b3e7-ab5c43dd3950,0fb51040-c70d-415e-9a90-9d6766f90ffc,a1,[Yes-No-Question]
2,011bb520-7041-704f-b3e7-ab5c43dd3950,10f45c05-44af-4afb-ac8c-e8fe0411e9a5,a1,"[Statement-non-opinion, Statement-non-opinion,..."


In [18]:

# ============================================================
# 3) Definir eixos GLOBAIS: states (1ª ordem) e superstates (2ª ordem)
# ============================================================
# states: todas as labels observadas
global_states = sorted(df[label_col].dropna().unique().tolist())
state_index = {s: i for i, s in enumerate(global_states)}

# superstates: todos os pares consecutivos observados no dataset (sem produto cartesiano completo para evitar explosão)
super_set = set()
for seq in seq_by_chat["sequence"]:
    for a, b in pairwise(seq):
        super_set.add((a, b))

global_superstates = sorted(super_set)
super_index = {ab: i for i, ab in enumerate(global_superstates)}

print(f"Total de states (1ª ordem): {len(global_states)}")
print(f"Total de superstates (2ª ordem): {len(global_superstates)}")


Total de states (1ª ordem): 29
Total de superstates (2ª ordem): 276


In [19]:

# ============================================================
# 4) Parâmetros
# ============================================================
TOP_K_USERS = None          # defina None para todos os usuários
MIN_SEQ_LEN = 2           # ignora chats com menos que isso


In [20]:

# ============================================================
# 5) Utilitários para montar matrizes nas ORDENS GLOBAIS
# ============================================================
import numpy as np

def first_order_matrix_global(sequences, global_states, state_index):
    n = len(global_states)
    M = np.zeros((n, n), dtype=int)
    # contar transições
    for seq in sequences:
        for a, b in pairwise(seq):
            if a in state_index and b in state_index:
                M[state_index[a], state_index[b]] += 1
    # normalizar por linha
    row_sums = M.sum(axis=1, keepdims=True)
    with np.errstate(divide='ignore', invalid='ignore'):
        P = M / row_sums
        P = np.nan_to_num(P, nan=0.0)
    edges = int(M.sum())
    return M, P, edges

def second_order_matrix_global(sequences, global_states, state_index, global_superstates, super_index):
    r = len(global_superstates)
    c = len(global_states)
    M = np.zeros((r, c), dtype=int)
    for seq in sequences:
        # precisamos de trios para 2ª ordem, mas as linhas são pares (a,b) globais
        from itertools import tee
        a, b, c_iter = tee(seq, 3)
        next(b, None)
        next(c_iter, None); next(c_iter, None)
        for a1, b1, c1 in zip(a, b, c_iter):
            ab = (a1, b1)
            if ab in super_index and c1 in state_index:
                M[super_index[ab], state_index[c1]] += 1
    row_sums = M.sum(axis=1, keepdims=True)
    with np.errstate(divide='ignore', invalid='ignore'):
        P = M / row_sums
        P = np.nan_to_num(P, nan=0.0)
    edges = int(M.sum())
    return M, P, edges


In [21]:

# ============================================================
# 6) Selecionar usuários e processar
# ============================================================
# Filtrar sequências curtas
seq_by_chat = seq_by_chat[seq_by_chat["sequence"].apply(len) >= MIN_SEQ_LEN].copy()

# Ranking por volume
user_topic_stats = (seq_by_chat.groupby(["userId", "topic"])["sequence"]
                         .agg(chats="count",
                              total_labels=lambda s: int(sum(len(x) for x in s)))
                         .reset_index()
                         .sort_values("total_labels", ascending=False))




if TOP_K_USERS is not None:
    user_topic_stats = user_topic_stats.head(TOP_K_USERS)

user_topic_pairs = user_topic_stats[["userId", "topic"]].drop_duplicates().values.tolist()
print("Usuário + tópico únicos:", len(user_topic_pairs))
user_topic_stats.head(10)


Usuário + tópico únicos: 916


,userId,topic,chats,total_labels
475,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,2,125
458,81cbb5e0-9011-7077-7e4e-7aca1564666c,a6,8,115
799,e12ba500-f0e1-70f2-73ab-05c1cbc44b76,a3,11,104
893,f16b2520-6021-70ee-b980-7a50d804795d,a5,9,102
52,01eb0530-d011-70d4-70a6-3d8b36a0733f,a6,11,101
645,b19b0560-c011-70d1-0a4f-c32cfa1ae4cd,a3,2,96
605,a1fb3570-f021-705f-6d23-353ec8617ab4,a3,4,85
709,c16be560-e061-709d-a447-5a0779910084,a7,3,85
102,117bf560-5011-70e9-de01-b833f5651c83,a7,7,82
892,f16b2520-6021-70ee-b980-7a50d804795d,a4,11,79


In [22]:
# ============================================================
# 7) Plot fixo por usuário + tópico (mesmos eixos para todos)
# ============================================================
import os, re
import numpy as np
import matplotlib.pyplot as plt

# Lista de pares (userId, topic) a processar — usa a ordem de user_topic_stats
user_topic_pairs = user_topic_stats[["userId", "topic"]].drop_duplicates().values.tolist()

summary_rows = []
total_pairs = len(user_topic_pairs)

for i, (uid, tpc) in enumerate(user_topic_pairs, start=1):
    # Sequências do par (usuário, tópico)
    user_topic_seqs = seq_by_chat[(seq_by_chat["userId"] == uid) & (seq_by_chat["topic"] == tpc)]["sequence"].tolist()
    if not user_topic_seqs:
        continue

    # Nome seguro para arquivos
    topic_safe = re.sub(r"[^A-Za-z0-9_\-]+", "_", str(tpc))

    # -------- 1ª ordem --------
    M1, P1, e1 = first_order_matrix_global(user_topic_seqs, global_states, state_index)
    np.savez(os.path.join(out_dir, f"user_{uid}_topic_{topic_safe}_order1.npz"),
             P=P1, M=M1,
             states=np.array(global_states, dtype=object),
             userId=uid, topic=tpc, order=1)

    fig = plt.figure(figsize=(10, 8))
    plt.imshow(P1, aspect='auto', vmin=0, vmax=1)
    plt.title(f"User {uid} | Topic {tpc} — Markov 1º ordem (eixos globais)")
    plt.xlabel("Próximo estado")
    plt.ylabel("Estado atual")
    plt.xticks(range(len(global_states)), global_states, rotation=90)
    plt.yticks(range(len(global_states)), global_states)
    plt.colorbar()
    plt.tight_layout()
    out1 = os.path.join(out_dir, f"user_{uid}_topic_{topic_safe}_order1_fixed.png")
    plt.savefig(out1, dpi=160)
    plt.close(fig)

    # -------- 2ª ordem --------
    M2, P2, e2 = second_order_matrix_global(
        user_topic_seqs, global_states, state_index, global_superstates, super_index
    )
    np.savez(os.path.join(out_dir, f"user_{uid}_topic_{topic_safe}_order2.npz"),
             P=P2, M=M2,
             states=np.array(global_states, dtype=object),
             superstates=np.array(global_superstates, dtype=object),
             userId=uid, topic=tpc, order=2)

    fig = plt.figure(figsize=(12, 9))
    plt.imshow(P2, aspect='auto', vmin=0, vmax=1)
    plt.title(f"User {uid} | Topic {tpc} — Markov 2º ordem (eixos globais)")
    plt.xlabel("Próximo estado")
    plt.ylabel("Par de estados (histórico) — eixos globais")

    # X ticks
    plt.xticks(range(len(global_states)), global_states, rotation=90)
    # Y ticks (subamostrar se houver muitos pares)
    max_ticks = 100
    if len(global_superstates) > max_ticks:
        step = max(1, len(global_superstates) // max_ticks)
        yticks = list(range(0, len(global_superstates), step))
        ylabels = [f"{a}|{b}" for (a, b) in [global_superstates[k] for k in yticks]]
    else:
        yticks = list(range(len(global_superstates)))
        ylabels = [f"{a}|{b}" for (a, b) in global_superstates]

    plt.yticks(yticks, ylabels)
    plt.colorbar()
    plt.tight_layout()
    out2 = os.path.join(out_dir, f"user_{uid}_topic_{topic_safe}_order2_fixed.png")
    plt.savefig(out2, dpi=160)
    plt.close(fig)

    summary_rows.append({
        "userId": uid,
        "topic": tpc,
        "order1_edges": e1,
        "order2_edges": e2,
        "order1_path": out1,
        "order2_path": out2
    })

    if i % 10 == 0:
        print(f"Processados {i}/{total_pairs} pares usuário+tópico...")

summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(out_dir, "per_user_topic_fixed_summary.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8")
print("Resumo salvo em:", summary_csv)

summary_df.head(10)


Processados 10/916 pares usuário+tópico...
Processados 20/916 pares usuário+tópico...
Processados 30/916 pares usuário+tópico...
Processados 40/916 pares usuário+tópico...
Processados 50/916 pares usuário+tópico...
Processados 60/916 pares usuário+tópico...
Processados 70/916 pares usuário+tópico...
Processados 80/916 pares usuário+tópico...
Processados 90/916 pares usuário+tópico...
Processados 100/916 pares usuário+tópico...
Processados 110/916 pares usuário+tópico...
Processados 120/916 pares usuário+tópico...
Processados 130/916 pares usuário+tópico...
Processados 140/916 pares usuário+tópico...
Processados 150/916 pares usuário+tópico...
Processados 160/916 pares usuário+tópico...
Processados 170/916 pares usuário+tópico...
Processados 180/916 pares usuário+tópico...
Processados 190/916 pares usuário+tópico...
Processados 200/916 pares usuário+tópico...
Processados 210/916 pares usuário+tópico...
Processados 220/916 pares usuário+tópico...
Processados 230/916 pares usuário+tópico.

,userId,topic,order1_edges,order2_edges,order1_path,order2_path
0,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,123,121,da_artifacts/user_81fbe5c0-8001-7054-3ac3-3e9d...,da_artifacts/user_81fbe5c0-8001-7054-3ac3-3e9d...
1,81cbb5e0-9011-7077-7e4e-7aca1564666c,a6,107,99,da_artifacts/user_81cbb5e0-9011-7077-7e4e-7aca...,da_artifacts/user_81cbb5e0-9011-7077-7e4e-7aca...
2,e12ba500-f0e1-70f2-73ab-05c1cbc44b76,a3,93,82,da_artifacts/user_e12ba500-f0e1-70f2-73ab-05c1...,da_artifacts/user_e12ba500-f0e1-70f2-73ab-05c1...
3,f16b2520-6021-70ee-b980-7a50d804795d,a5,93,84,da_artifacts/user_f16b2520-6021-70ee-b980-7a50...,da_artifacts/user_f16b2520-6021-70ee-b980-7a50...
4,01eb0530-d011-70d4-70a6-3d8b36a0733f,a6,90,79,da_artifacts/user_01eb0530-d011-70d4-70a6-3d8b...,da_artifacts/user_01eb0530-d011-70d4-70a6-3d8b...
5,b19b0560-c011-70d1-0a4f-c32cfa1ae4cd,a3,94,92,da_artifacts/user_b19b0560-c011-70d1-0a4f-c32c...,da_artifacts/user_b19b0560-c011-70d1-0a4f-c32c...
6,a1fb3570-f021-705f-6d23-353ec8617ab4,a3,81,77,da_artifacts/user_a1fb3570-f021-705f-6d23-353e...,da_artifacts/user_a1fb3570-f021-705f-6d23-353e...
7,c16be560-e061-709d-a447-5a0779910084,a7,82,79,da_artifacts/user_c16be560-e061-709d-a447-5a07...,da_artifacts/user_c16be560-e061-709d-a447-5a07...
8,117bf560-5011-70e9-de01-b833f5651c83,a7,75,68,da_artifacts/user_117bf560-5011-70e9-de01-b833...,da_artifacts/user_117bf560-5011-70e9-de01-b833...
9,f16b2520-6021-70ee-b980-7a50d804795d,a4,68,57,da_artifacts/user_f16b2520-6021-70ee-b980-7a50...,da_artifacts/user_f16b2520-6021-70ee-b980-7a50...


In [23]:

# ============================================================
# 8) Exportar ordens globais (para referência/comparabilidade)
# ============================================================
import json
axes_meta = {
    "states_global": global_states,
    "superstates_global": [[a, b] for (a,b) in global_superstates]
}
with open(os.path.join(out_dir, "axes_global.json"), "w", encoding="utf-8") as f:
    json.dump(axes_meta, f, ensure_ascii=False, indent=2)

print("Eixos globais exportados em:", os.path.join(out_dir, "axes_global.json"))


Eixos globais exportados em: da_artifacts/axes_global.json
